# 第 3 章 线性模型（下）：Softmax 回归多分类

Logistic 回归可以有效地解决二分类问题，但更广泛的分类任务是**多分类**：类别数 $C > 2$。**Softmax 回归**就是 Logistic 回归在多分类问题上的推广。

在 Softmax 回归中，类别标签 $y\in\{1,2,\ldots,C\}$。给定一个样本 $\boldsymbol{x}$，使用 Softmax 回归预测的属于类别 $c$ 的条件概率为

$$ p(y=c\mid \boldsymbol{x}) = \operatorname{softmax}(\boldsymbol{w}_c^\top \boldsymbol{x}+b_c), $$

其中 $\boldsymbol{w}_c$ 是第 $c$ 类的权重向量，$b_c$ 是第 $c$ 类的偏置。

本节实现 Softmax 回归模型，先在合成三分类数据集 Multi1000 上演示，再做鸢尾花分类。


## 1. 构建 Multi1000 数据集

构建一个简单的 3 分类数据集 **Multi1000**：1000 条样本，每个样本 2 个特征。本任务的数据来自 3 个不同的簇，每个簇对应一个类别。


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt


def make_multiclass_classification(n_samples=100, n_features=2, n_classes=3, shuffle=True, noise=0.1, seed=None):
    """生成多类别分类数据。"""
    # 计算每个类别的样本数
    g = torch.Generator().manual_seed(seed) if seed is not None else None
    n_per_class = [n_samples // n_classes] * n_classes
    for i in range(n_samples - sum(n_per_class)):
        n_per_class[i % n_classes] += 1

    X = torch.zeros(n_samples, n_features)
    y = torch.zeros(n_samples, dtype=torch.int64)

    # 随机生成簇中心
    centroids = torch.randperm(2 ** n_features, generator=g)[:n_classes]
    centroids_bin = np.unpackbits(centroids.numpy().astype('uint8')).reshape(-1, 8)[:, -n_features:]
    centroids = torch.tensor(centroids_bin, dtype=torch.float32)
    centroids = 1.5 * centroids - 1                      # 控制簇中心分离程度

    # 初始化特征
    X[:, :n_features] = torch.randn(n_samples, n_features, generator=g)

    stop = 0
    for k, centroid in enumerate(centroids):
        start, stop = stop, stop + n_per_class[k]
        y[start:stop] = k % n_classes
        X_k = X[start:stop, :n_features]
        A = 2 * torch.rand(n_features, n_features, generator=g) - 1   # 控制每类分散程度
        X_k = X_k @ A + centroid
        X[start:stop, :n_features] = X_k

    if noise > 0:
        # 随机给一部分样本换标签当成噪声
        noise_mask = torch.rand(n_samples, generator=g) < noise
        n_noisy = int(noise_mask.sum())
        y[noise_mask] = torch.randint(0, n_classes, (n_noisy,), generator=g)
    if shuffle:
        idx = torch.randperm(n_samples, generator=g)
        X = X[idx]; y = y[idx]
    return X, y


torch.manual_seed(102)
n_samples = 1000
X, y = make_multiclass_classification(n_samples=n_samples, n_features=2, n_classes=3, noise=0.2)

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0].tolist(), X[:, 1].tolist(), marker='*', c=y.tolist())
plt.title('Multi1000'); plt.show()


In [ ]:
# 拆分 640 / 160 / 200
num_train, num_dev, num_test = 640, 160, 200
X_train, y_train = X[:num_train], y[:num_train]
X_dev,   y_dev   = X[num_train:num_train + num_dev], y[num_train:num_train + num_dev]
X_test,  y_test  = X[num_train + num_dev:], y[num_train + num_dev:]

print('X_train shape:', X_train.shape, 'y_train shape:', y_train.shape)
print('前 5 个标签:', y_train[:5])


## 2. Softmax 函数

对于 $K$ 维向量 $\boldsymbol{x}=[x_1,\ldots,x_K]$，Softmax 计算公式为

$$ \operatorname{softmax}(x_k) = \frac{\exp(x_k)}{\sum_{i=1}^K \exp(x_i)}. $$

实现 Softmax 时要注意**上溢出**和**下溢出**问题：

- **上溢出**：当 $\boldsymbol{x}$ 中元素是非常大的正数时，$\exp(\boldsymbol{x})$ 会溢出；
- **下溢出**：当 $\boldsymbol{x}$ 中元素都是非常大的负数时，$\exp(\boldsymbol{x})$ 会被四舍五入为 0，导致分母为 0。

解决办法：用 $x_k - \max(\boldsymbol{x})$ 代替 $x_k$。减去最大值后，$x_k$ 最大为 0，避免了上溢；同时分母中至少有一个 $\exp(0)=1$ 的加和项，避免了下溢。

> PyTorch 内置的 `torch.softmax` / `F.softmax` 都已经做了这个数值稳定化处理，这里我们手写一遍是为了理解原理。


In [ ]:
def softmax(X):
    """数值稳定的 softmax，X 形状 [N, D]。"""
    x_max = X.max(dim=1, keepdim=True).values
    x_exp = torch.exp(X - x_max)
    partition = x_exp.sum(dim=1, keepdim=True)
    return x_exp / partition


# 观察 softmax 的计算方式
X_demo = torch.tensor([[0.1, 0.2, 0.3, 0.4]])
print(softmax(X_demo))


## 3. Softmax 回归模型

以批量形式定义 Softmax 回归：将 $N$ 个样本归为一组，令 $\boldsymbol{X}\in\mathbb{R}^{N\times D}$ 为特征矩阵，则

$$ \hat{\boldsymbol{Y}} = \operatorname{softmax}(\boldsymbol{X}\boldsymbol{W}+\boldsymbol{b}), $$

其中 $\boldsymbol{W}\in\mathbb{R}^{D\times C}$ 为权重矩阵（每列对应一个类别），$\boldsymbol{b}\in\mathbb{R}^C$ 为偏置。矩阵 $\boldsymbol{X}\boldsymbol{W}$ 和向量 $\boldsymbol{b}$ 按广播机制相加。

我们沿用第 1 章 `Op` 基类的接口实现。


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))

from nndl import Op


class Model_SR(Op):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.params = {
            'W': torch.zeros(input_size, output_size),
            'b': torch.zeros(output_size),
        }
        # 存放参数的梯度
        self.grads = {}
        self.X = None
        self.outputs = None
        self.output_size = output_size

    def forward(self, inputs):
        """X: (N, D) -> output: (N, C)"""
        self.X = inputs
        score = inputs @ self.params['W'] + self.params['b']
        self.outputs = softmax(score)
        return self.outputs

    def backward(self, labels):
        """手动给出 loss 关于参数的梯度（即 softmax + 交叉熵的合成梯度）。"""
        N = labels.shape[0]
        # 把标签转 one-hot
        y_onehot = torch.nn.functional.one_hot(labels, num_classes=self.output_size).float()
        # ∂R/∂W = -1/N * X^T (y - y_hat)
        self.grads['W'] = -1 / N * (self.X.t() @ (y_onehot - self.outputs))
        # ∂R/∂b = -1/N * 1^T (y - y_hat)
        self.grads['b'] = -1 / N * (y_onehot - self.outputs).sum(dim=0)


# 测试一下：随机生成 1 条长度为 4 的数据
torch.manual_seed(0)
inputs = torch.randn(1, 4)
model = Model_SR(input_size=4, output_size=3)
print('Input :', inputs)
print('Output:', model(inputs))


从输出结果看，采用全 0 初始化后，属于每个类别的条件概率均为 $\frac{1}{C}$。这是因为线性函数的输出恒为 0，Softmax 之后每类概率相等。


## 4. 损失函数：多类交叉熵

对于 $N$ 个样本，令 $\hat{\boldsymbol{y}}^{(n)} = \operatorname{softmax}(\boldsymbol{W}^\top \boldsymbol{x}^{(n)}+\boldsymbol{b})$。多分类交叉熵损失定义为

$$ \mathcal{R}(\boldsymbol{W},\boldsymbol{b}) = -\frac{1}{N}\sum_{n=1}^N (\boldsymbol{y}^{(n)})^\top\log\hat{\boldsymbol{y}}^{(n)}, $$

由于 $\boldsymbol{y}^{(n)}$ 是 one-hot，只有正确类别那一维是 1，所以可以等价写成

$$ \mathcal{R}(\boldsymbol{W},\boldsymbol{b}) = -\frac{1}{N}\sum_{n=1}^N \log [\hat{\boldsymbol{y}}^{(n)}]_{y^{(n)}}. $$

即"交叉熵损失只关心正确类别的预测概率"。


In [ ]:
class MultiCrossEntropyLoss(Op):
    def __init__(self):
        self.predicts = None
        self.labels = None

    def __call__(self, predicts, labels):
        return self.forward(predicts, labels)

    def forward(self, predicts, labels):
        """predicts (N, C)、labels (N,) 长整型类别索引。"""
        self.predicts = predicts
        self.labels = labels
        N = predicts.shape[0]
        eps = 1e-12
        # 取正确类别对应的预测概率
        p_correct = predicts[torch.arange(N), labels].clamp(eps, 1.0)
        return -torch.log(p_correct).mean()


# 测试一下：用上面的 outputs 和 label 0
mce_loss = MultiCrossEntropyLoss()
print('loss:', mce_loss(model.outputs, torch.tensor([0])))


## 5. 复用 `RunnerV2` 与 `accuracy`、`SimpleBatchGD`

`RunnerV2`、`SimpleBatchGD`、`accuracy` 都在上一节实现并讲解过，本节直接从 `nndl` 工具包导入。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))

from nndl import RunnerV2, Optimizer, SimpleBatchGD, accuracy

## 6. 模型训练（Multi1000）

实例化 `RunnerV2` 类，使用训练集和验证集训练 500 个回合，每 100 个回合打印一次。


In [ ]:
torch.manual_seed(0)
model = Model_SR(input_size=2, output_size=3)
optimizer = SimpleBatchGD(init_lr=0.1, model=model)
loss_fn = MultiCrossEntropyLoss()
metric = accuracy

runner = RunnerV2(model, optimizer, metric, loss_fn)
runner.train([X_train, y_train], [X_dev, y_dev],
             num_epochs=500, log_epochs=100,
             save_path='./checkpoint/model_sr_best.pt')


# 可视化训练曲线
def plot(runner):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(runner.history['train_score'], color='red',  label='Train accuracy')
    plt.plot(runner.history['dev_score'],   color='blue', label='Dev accuracy')
    plt.ylabel('score'); plt.xlabel('epoch'); plt.legend(loc='lower right')
    plt.subplot(1, 2, 2)
    plt.plot(runner.history['train_loss'], color='red',  label='Train loss')
    plt.plot(runner.history['dev_loss'],   color='blue', label='Dev loss')
    plt.ylabel('loss'); plt.xlabel('epoch'); plt.legend(loc='upper right')
    plt.tight_layout(); plt.show()


plot(runner)


## 7. 模型评价


In [ ]:
score, loss = runner.evaluate([X_test, y_test])
print(f'[Test] score / loss: {score:.4f} / {loss:.4f}')


## 8. 实践：基于 Softmax 回归的鸢尾花分类

入门机器学习的经典实验之一：用 sklearn 自带的 Iris 数据集做三分类。

**Iris 数据集**包含 3 种鸢尾花类别（Setosa、Versicolour、Virginica），每种 50 个样本，共 150 个样本。每个样本有 4 个属性：花萼长度、花萼宽度、花瓣长度、花瓣宽度。

| 类别 | 标签 |
|---|---|
| Setosa | 0 |
| Versicolour | 1 |
| Virginica | 2 |

机器学习实践流程主要包括 8 个步骤：数据处理、模型构建、损失函数定义、优化器构建、评价指标定义、模型训练、模型评价、模型预测。其中前 5 个步骤可以看作模型准备阶段。


### 8.1 数据加载与归一化

将 150 条数据按 8:1:1 划分为训练集、验证集和测试集。归一化把每个特征缩放到 $[0, 1]$。


In [ ]:
from sklearn.datasets import load_iris
import numpy as np


def load_data(shuffle=True):
    iris = load_iris()
    X = torch.tensor(iris.data, dtype=torch.float32)
    y = torch.tensor(iris.target, dtype=torch.int64)

    # 归一化
    X_min = X.min(dim=0).values
    X_max = X.max(dim=0).values
    X = (X - X_min) / (X_max - X_min)

    if shuffle:
        idx = torch.randperm(X.shape[0])
        X = X[idx]; y = y[idx]
    return X, y


torch.manual_seed(102)
num_train, num_dev, num_test = 120, 15, 15
X, y = load_data(shuffle=True)
X_train, y_train = X[:num_train], y[:num_train]
X_dev,   y_dev   = X[num_train:num_train + num_dev], y[num_train:num_train + num_dev]
X_test,  y_test  = X[num_train + num_dev:], y[num_train + num_dev:]

print('X_train:', X_train.shape, ' y_train:', y_train.shape)
print('前 5 个标签:', y_train[:5])


### 8.2 模型构建 + 训练

输入维度 4，输出类别数 3。


In [ ]:
torch.manual_seed(0)
model = Model_SR(input_size=4, output_size=3)
optimizer = SimpleBatchGD(init_lr=0.2, model=model)
loss_fn = MultiCrossEntropyLoss()
metric = accuracy

runner = RunnerV2(model, optimizer, metric, loss_fn)
runner.train([X_train, y_train], [X_dev, y_dev],
             num_epochs=80, log_epochs=10,
             save_path='./checkpoint/model_iris_best.pt')

plot(runner)


### 8.3 模型评价


In [ ]:
runner.load_model('./checkpoint/model_iris_best.pt')
score, loss = runner.evaluate([X_test, y_test])
print(f'[Test] score / loss: {score:.4f} / {loss:.4f}')


### 8.4 单点预测

由于 `Model_SR` 接收的是一组样本（$N \times D$ 的矩阵），而预测时通常只输入一条数据（$D$ 维向量），需要用 `unsqueeze(0)` 给特征向量增加新的第 0 维，把它变成 $1 \times D$ 的矩阵。


In [ ]:
# 取测试集第一条样本
x0 = X_test[0].unsqueeze(0)
label = y_test[0].item()

predicts = runner.predict(x0)
pred = predicts.argmax(dim=1).item()
print(f'真实类别: {label}    预测类别: {pred}')


## 小结

本章实现了 **Logistic 回归**和 **Softmax 回归**两种基本的线性分类模型，并实现了基于梯度下降的参数学习方法。为了让训练框架支持梯度下降法，我们在 `RunnerV1` 的基础上构建了 **`RunnerV2`**，它还引入了**早停法**：训练过程中保存验证集上最优的模型，缓解过拟合。

在实践部分，我们基于 `RunnerV2` 跑通了 Softmax 回归的鸢尾花分类，覆盖了机器学习实践的 8 个主要步骤：数据处理、模型构建、损失函数定义、优化器构建、评价指标定义、模型训练、模型评价、模型预测。在后续章节中，我们都会遵循这个实践范式。
